In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import * 

In [0]:
spark = SparkSession.builder.appName("ECommerceDataPipeline").getOrCreate()

In [0]:
usersDF = spark.table("ecommerce_fashion.bronze.users")

In [0]:
usersDF.show(5)

In [0]:
usersDF.printSchema()

In [0]:
#Normalize countryCode to upperCase
usersDF = usersDF.withColumn('countryCode', upper(col('countryCode')))
usersDF.select(col('countryCode')).show(5)

In [0]:
#Make the language column from two letters to the full word using the expr function

usersDF.select(col("language")).distinct().show()

In [0]:
usersDF = usersDF.withColumn("language", expr("case when language = 'en' then 'english' when language = 'fr' then 'french' when language = 'de' then 'german' when language = 'it' then 'italian' when language = 'es' then 'spanish' else language end"))
usersDF.select(col("language")).distinct().show()



In [0]:
usersDF.show(5)

In [0]:
#Correcting and expanding potential data entry errors in gender column

usersDF = usersDF.withColumn("gender", when(col("gender").startswith("M"), "Male" ).
                                       when(col("gender").startswith("F"), "Female").
                                       otherwise("other"))

In [0]:
usersDF.select("civilityTitle").distinct().show()

In [0]:
#Using regex correct civility titles

usersDF = usersDF.withColumn("civilityTitle", regexp_replace("civilityTitle", "(miss|mrs)", "Ms"))

In [0]:
usersDF.show(5)

In [0]:
#Derive a new column yearssincelastlogin from column daysSinceLastLogin

usersDF = usersDF.withColumn("yearsSinceLastLogin", round(col("daysSinceLastLogin")/365, 1))

In [0]:
usersDF.show(5)

In [0]:
usersDF.select(col("seniorityAsYears")).distinct().show()


In [0]:
#Calculate the age of account in years and categorize into 'account_age_group'

usersDF = usersDF.withColumn("account_age_group", when(col("seniorityAsYears").cast("double") < 1, "New").
                                                when(col("seniorityAsYears").cast("double") > 5, "Experienced").
                                                otherwise("Intermediate"))

In [0]:
#Add a new column current_year for comparision

usersDF = usersDF.withColumn("current_year", year(current_date()))

In [0]:
#Add a new column user descriptor by creatively combining multiple columns gender, country code, civility title and language

usersDF = usersDF.withColumn("user_descriptor", lower(concat(
    col("gender"), lit("_"),
    col("countryCode"), lit("_"),
    col("civilityTitle"), lit("_"),
    col("language")
)))

In [0]:
usersDF = usersDF.withColumn("flag_long_title", length(col("civilitytitle")) > 10)

In [0]:
usersDF.printSchema()

In [0]:
usersDF.show(5)

In [0]:
integer_cols_users = ["socialNbFollowers", "socialNbFollows", "socialProductsLiked", "productsListed", "productsSold", 
                      "productsWished","productsBought", "civilityGenderId", "daysSinceLastLogin", "seniority", "current_year"]

decimal_cols_users = ["productsPassRate", "seniorityAsMonths", "seniorityAsYears", "yearsSinceLastLogin", ]

boolean_cols_users = ["hasAnyApp", "hasAndroidApp", "hasIosApp", "hasProfilePicture"]


for items in integer_cols_users:
    usersDF = usersDF.withColumn(items, col(items).cast(IntegerType()))

for items in decimal_cols_users:
    usersDF = usersDF.withColumn(items, col(items).cast(DecimalType(10,2)))

for items in boolean_cols_users:
    usersDF = usersDF.withColumn(items, col(items).cast("boolean"))

In [0]:
usersDF = usersDF.withColumn("dayssincelastlogin",
                             when(col("dayssincelastlogin").isNotNull(),
                                  col("dayssincelastlogin").cast(IntegerType()))
                             .otherwise(0))

In [0]:
usersDF.write \
    .format("delta")\
    .mode("overwrite")\
    .option("overwrite_schema", "true")\
    .saveAsTable("ecommerce_fashion.silver.users")

In [0]:
buyersDF = spark.table("ecommerce_fashion.bronze.buyers")


In [0]:
decimal_cols = ["topbuyerratio","femalebuyersratio","topfemalebuyersratio", "boughtperwishlistratio", "boughtperlikeratio", "topboughtperlikeratio",  "meanproductsbought", "meanproductswished", "meanproductsliked", "topmeanproductsbought", "topmeanproductswished", "topmeanproductsliked", "meanofflinedays", "meanfollowers", "meanfollowing", "topmeanfollowers", "topmeanfollowing" ]

In [0]:
buyersDF.printSchema()

In [0]:
integer_cols = ["buyers", "topbuyers", "femalebuyers", "malebuyers", "topfemalebuyers", "topmalebuyers", "totalproductsbought", "totalproductsliked","toptotalproductswished", "toptotalproductsliked", "totalproductswished" ]
decimal_cols = ["topbuyerratio","femalebuyersratio","topfemalebuyersratio", "boughtperwishlistratio", "boughtperlikeratio", "topboughtperlikeratio",  "meanproductsbought", "meanproductswished", "meanproductsliked", "topmeanproductsbought", "topmeanproductswished", "topmeanproductsliked", "meanofflinedays", "meanfollowers", "meanfollowing", "topmeanfollowers", "topmeanfollowing", "toptotalproductsbought" ]
for item in integer_cols:
  buyersDF = buyersDF.withColumn(item, col(item).cast(IntegerType()))

for item in decimal_cols:
  buyersDF = buyersDF.withColumn(item, col(item).cast(DecimalType(10,2)))
  

In [0]:
buyersDF = buyersDF.withColumn("country", initcap(col("country")))

for col_name in integer_cols:
    buyersDF = buyersDF.fillna({col_name: 0})

# Calculate the ratio of female to male buyers
buyersDF = buyersDF.withColumn("female_to_male_ratio", 
                               round(col("femalebuyers") / (col("malebuyers") + 1), 2))

# Determine the market potential by comparing wishlist and purchases
buyersDF = buyersDF.withColumn("wishlist_to_purchase_ratio", 
                               round(col("totalproductswished") / (col("totalproductsbought") + 1), 2))

# Tag countries with a high engagement ratio
high_engagement_threshold = 0.5
buyersDF = buyersDF.withColumn("high_engagement",
                               when(col("boughtperwishlistratio") > high_engagement_threshold, True)
                               .otherwise(False))
                               
    # Flag markets with increasing female buyer participation
buyersDF = buyersDF.withColumn("growing_female_market",
                               when(col("femalebuyersratio") > col("topfemalebuyersratio"), True)
                               .otherwise(False))

In [0]:
buyersDF.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("ecommerce_fashion.silver.buyers")

In [0]:
sellersDF = spark.table("ecommerce_fashion.bronze.sellers")
sellersDF.printSchema()


In [0]:
integer_colssellers = ["nbsellers", "totalproductssold","totalproductslisted", "totalbought", "totalwished", "totalproductsliked" ]
decimal_colssellers = ["meanproductssold", "meanproductslisted", "meansellerpassrate", "meanproductsbought", "meanproductswished", "meanproductsliked", "meanfollowers", "meanfollows", "percentofappusers", "percentofiosusers", "meanseniority"]
for items in integer_colssellers:
    sellersDF = sellersDF.withColumn(items, col(items).cast(IntegerType()))
for items in decimal_colssellers:
    sellersDF = sellersDF.withColumn(items, col(items).cast(DecimalType(10,2)))

In [0]:
sellersDF = sellersDF.withColumn("country", initcap(col("country"))) \
                                                .withColumn("sex", upper(col("sex")))


#Add a column to categorize the number of sellers
sellersDF = sellersDF.withColumn("seller_size_category", 
                               when(col("nbsellers") < 500, "Small") \
                               .when((col("nbsellers") >= 500) & (col("nbsellers") < 2000), "Medium") \
                               .otherwise("Large"))

# Calculate the mean products listed per seller as an indicator of seller activity
sellersDF = sellersDF.withColumn("mean_products_listed_per_seller", 
                               round(col("totalproductslisted") / col("nbsellers"), 2))

# Identify markets with high seller pass rate
sellersDF = sellersDF.withColumn("high_seller_pass_rate", 
                               when(col("meansellerpassrate") > 0.75, "High") \
                               .otherwise("Normal"))

mean_pass_rate = sellersDF.select(round(avg("meansellerpassrate"), 2).alias("avg_pass_rate")).collect()[0]["avg_pass_rate"]

sellersDF = sellersDF.withColumn("meansellerpassrate",
                                 when(col("meansellerpassrate").isNull(), mean_pass_rate)
                                 .otherwise(col("meansellerpassrate")))

In [0]:
sellersDF.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("ecommerce_fashion.silver.sellers")

In [0]:
countriesDF = spark.table("ecommerce_fashion.bronze.countries")
countriesDF.printSchema()

In [0]:
countriesDF = countriesDF \
    .withColumn("sellers", col("sellers").cast(IntegerType())) \
    .withColumn("topsellers", col("topsellers").cast(IntegerType())) \
    .withColumn("topsellerratio", col("topsellerratio").cast(DecimalType(10, 2))) \
    .withColumn("femalesellersratio", col("femalesellersratio").cast(DecimalType(10, 2))) \
    .withColumn("topfemalesellersratio", col("topfemalesellersratio").cast(DecimalType(10, 2))) \
    .withColumn("femalesellers", col("femalesellers").cast(IntegerType())) \
    .withColumn("malesellers", col("malesellers").cast(IntegerType())) \
    .withColumn("topfemalesellers", col("topfemalesellers").cast(IntegerType())) \
    .withColumn("topmalesellers", col("topmalesellers").cast(IntegerType())) \
    .withColumn("countrysoldratio", col("countrysoldratio").cast(DecimalType(10, 2))) \
    .withColumn("bestsoldratio", col("bestsoldratio").cast(DecimalType(10, 2))) \
    .withColumn("toptotalproductssold", col("toptotalproductssold").cast(IntegerType())) \
    .withColumn("totalproductssold", col("totalproductssold").cast(IntegerType())) \
    .withColumn("toptotalproductslisted", col("toptotalproductslisted").cast(IntegerType())) \
    .withColumn("totalproductslisted", col("totalproductslisted").cast(IntegerType())) \
    .withColumn("topmeanproductssold", col("topmeanproductssold").cast(DecimalType(10, 2))) \
    .withColumn("topmeanproductslisted", col("topmeanproductslisted").cast(DecimalType(10, 2))) \
    .withColumn("meanproductssold", col("meanproductssold").cast(DecimalType(10, 2))) \
    .withColumn("meanproductslisted", col("meanproductslisted").cast(DecimalType(10, 2))) \
    .withColumn("meanofflinedays", col("meanofflinedays").cast(DecimalType(10, 2))) \
    .withColumn("topmeanofflinedays", col("topmeanofflinedays").cast(DecimalType(10, 2))) \
    .withColumn("meanfollowers", col("meanfollowers").cast(DecimalType(10, 2))) \
    .withColumn("meanfollowing", col("meanfollowing").cast(DecimalType(10, 2))) \
    .withColumn("topmeanfollowers", col("topmeanfollowers").cast(DecimalType(10, 2))) \
    .withColumn("topmeanfollowing", col("topmeanfollowing").cast(DecimalType(10, 2)))

countriesDF = countriesDF.withColumn("country", initcap(col("country")))


# Calculating the ratio of top sellers to total sellers
countriesDF = countriesDF.withColumn("top_seller_ratio", 
                                        round(col("topsellers") / col("sellers"), 2))

# countriesDF countries with a high ratio of female sellers
countriesDF = countriesDF.withColumn("high_female_seller_ratio", 
                                        when(col("femalesellersratio") > 0.5, True).otherwise(False))

# Adding a performance indicator based on the sold/listed ratio
countriesDF = countriesDF.withColumn("performance_indicator", 
                                        round(col("toptotalproductssold") / (col("toptotalproductslisted") + 1), 2))

# Flag countries with exceptionally high performance
performance_threshold = 0.8
countriesDF = countriesDF.withColumn("high_performance", 
                                        when(col("performance_indicator") > performance_threshold, True).otherwise(False))

countriesDF = countriesDF.withColumn("activity_level",
                                       when(col("meanofflinedays") < 30, "Highly Active")
                                       .when((col("meanofflinedays") >= 30) & (col("meanofflinedays") < 60), "Moderately Active")
                                       .otherwise("Low Activity"))

In [0]:
countriesDF.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("ecommerce_fashion.silver.countries")